# smolagents Agentic Architecture — From One Agent to Multi-Agent

A beginner-friendly, hands-on notebook using Hugging Face **smolagents**.

By the end, you will be able to:

- explain the parts of an agentic system,
- create custom tools with `@tool`,
- build a `CodeAgent`,
- compare `CodeAgent` with `ToolCallingAgent`,
- inspect an agent run, and
- build a manager–specialist multi-agent architecture.

> **Note:** The package is named `smolagents` (without a second “l”).

## 1. What is an agent?

A normal LLM receives a prompt and produces text. An **agent** can also decide to use tools, observe their results, and continue working until it reaches an answer.

### Core architecture

| Component | Job | Example in this notebook |
|---|---|---|
| User task | Defines the goal | “Calculate the final price of 3 keyboards” |
| Model | Reasons and selects actions | GPT-4o mini through OpenRouter |
| Agent loop | Repeats thought → action → observation | `CodeAgent` / `ToolCallingAgent` |
| Tools | Perform reliable external work | inventory, pricing, shipping |
| Memory | Stores steps and observations | `agent.memory` |
| Final answer | Returns the completed result | itemized quotation |

### Execution flow

**Task → Model decides → Tool executes → Observation returns → Model checks progress → Final answer**

If the task is not complete, the agent repeats the middle steps. This is a multi-step agent loop.

## 2. Install the required packages

Run this cell once in Google Colab. The `litellm` extra lets smolagents connect to OpenRouter and many other model providers.

In [1]:
!pip -q install -U "smolagents[litellm]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00


## 3. Connect the language model

This version uses an OpenRouter key and hides it while you type. You may replace the model ID with another tool-capable OpenRouter model.

In [2]:
import os
from getpass import getpass

from smolagents import CodeAgent, LiteLLMModel, ToolCallingAgent, tool

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: " )

model = LiteLLMModel(
    model_id="openrouter/openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",
    temperature=0.1,
)

print("Model is ready.")

Enter your OpenRouter API key: ··········
Model is ready.


## 4. Create the tools

A tool should do one clear job. Its name, type hints, docstring, argument descriptions, and return type help the model understand when and how to use it.

We use local sample data so the architecture remains easy to run and explain.

In [14]:
PRODUCTS = {
    "keyboard": {"price": 2500.0, "stock": 8},
    "mouse": {"price": 1200.0, "stock": 15},
    "monitor": {"price": 18000.0, "stock": 4},
    "webcam": {"price": 3200.0, "stock": 0},
}


@tool
def check_product(product_name: str) -> str:
    """Check a product's unit price and available stock.

    Args:
        product_name: Product name, such as keyboard, mouse, monitor, or webcam.
    """
    item = PRODUCTS.get(product_name.lower().strip())
    if item is None:
        return f"Product '{product_name}' was not found."
    return (
        f"Product: {product_name.lower()}, unit price: ₹{item['price']:.2f}, "
        f"stock: {item['stock']} units"
    )


@tool
def calculate_order(unit_price: float, quantity: int) -> str:
    """Calculate subtotal and quantity discount for an order.

    Orders of 3 to 4 units receive 5% off. Orders of 5 or more receive 10% off.

    Args:
        unit_price: Price of one unit in Indian rupees.
        quantity: Number of units requested.
    """
    if unit_price < 0 or quantity < 1:
        return "Invalid input: price must be non-negative and quantity must be at least 1."

    discount_rate = 0.10 if quantity >= 5 else 0.05 if quantity >= 3 else 0.0
    subtotal = unit_price * quantity
    discount = subtotal * discount_rate
    discounted_total = subtotal - discount

    return (
        f"Subtotal: ₹{subtotal:.2f}; discount: {discount_rate:.0%} "
        f"(₹{discount:.2f}); after discount: ₹{discounted_total:.2f}"
    )


@tool
def calculate_shipping(order_value: float, city: str) -> str:
    """Calculate shipping for an already-discounted order value.

    Shipping is free at ₹5,000 or above. Otherwise it is ₹150 for Kolkata,
    Bhubaneswar, and Delhi, and ₹250 for other cities.

    Args:
        order_value: Order value after discount, in Indian rupees.
        city: Delivery city.
    """
    if order_value >= 5000:
        fee = 0.0
    elif city.lower().strip() in {"kolkata", "bhubaneswar", "delhi"}:
        fee = 150.0
    else:
        fee = 250.0
    return f"Shipping fee to {city.title()}: ₹{fee:.2f}"

### Test tools directly first

A tool is just a callable component. Testing it separately makes debugging much easier than debugging the complete agent.

In [11]:
print(check_product("keyboard"))
print(calculate_order(unit_price=2500, quantity=3))
print(calculate_shipping(order_value=7125, city="Kolkata"))

Product: keyboard, unit price: ₹2500.00, stock: 8 units
Subtotal: ₹7500.00; discount: 5% (₹375.00); after discount: ₹7125.00
Shipping fee to Kolkata: ₹0.00


## 5. Build a CodeAgent

`CodeAgent` expresses its actions as small Python programs. This is useful when the model must combine tools, reuse values, perform calculations, or apply control flow.

`max_steps` prevents an endless loop. A lower value reduces cost; a higher value allows more complicated tasks.

In [13]:
sales_agent = CodeAgent(
    tools=[check_product, calculate_order, calculate_shipping],
    model=model,
    max_steps=6,
)

question = (
    "Tell me the weather of India usually."
)

answer = sales_agent.run(question)
print("\nFINAL ANSWER\n", answer)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me the weather of India usually.                                                                           │
│                                                                                                                 │
╰─ LiteLLMModel - openrouter/openai/gpt-4o-mini ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  weather_summary = {                                                                                              
      "Winter": "December to February, temperatures range from 5°C to 20°C in most parts, with northern regions    
  experiencing colder weather.",                                                                                   
      "Summer": "March to June, temperatures can soar above 40°C in many regions, especially in the northern       
  plains.",                                                                                                        
      "Monsoon": "June to September, characterized by heavy rainfall, particularly in the western coast and        
  northeastern states.",                                                                                           
      "Post-Monsoon": "October to November, temperatures start to cool down, and the weather becomes pleasant."    
  }                                                                                                                
                                                                                                                   
  final_answer(weather_summary)                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'Winter': 'December to February, temperatures range from 5°C to 20°C in most parts, with northern 
regions experiencing colder weather.', 'Summer': 'March to June, temperatures can soar above 40°C in many regions, 
especially in the northern plains.', 'Monsoon': 'June to September, characterized by heavy rainfall, particularly 
in the western coast and northeastern states.', 'Post-Monsoon': 'October to November, temperatures start to cool 
down, and the weather becomes pleasant.'}

[Step 1: Duration 2.89 seconds| Input tokens: 2,191 | Output tokens: 184]


FINAL ANSWER
 {'Winter': 'December to February, temperatures range from 5°C to 20°C in most parts, with northern regions experiencing colder weather.', 'Summer': 'March to June, temperatures can soar above 40°C in many regions, especially in the northern plains.', 'Monsoon': 'June to September, characterized by heavy rainfall, particularly in the western coast and northeastern states.', 'Post-Monsoon': 'October to November, temperatures start to cool down, and the weather becomes pleasant.'}


In [5]:
sales_agent = CodeAgent(
    tools=[check_product, calculate_order, calculate_shipping],
    model=model,
    max_steps=6,
)

question = (
    "A customer in Kolkata wants 3 keyboards. Check stock, apply the correct "
    "discount, calculate shipping, and give a clear final quotation. Do not "
    "promise the order if stock is insufficient."
)

answer = sales_agent.run(question)
print("\nFINAL ANSWER\n", answer)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ A customer in Kolkata wants 3 keyboards. Check stock, apply the correct discount, calculate shipping, and give  │
│ a clear final quotation. Do not promise the order if stock is insufficient.                                     │
│                                                                                                                 │
╰─ LiteLLMModel - openrouter/openai/gpt-4o-mini ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  product_name = "keyboard"                                                                                        
  stock_info = check_product(product_name)                                                                         
  print(stock_info)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Product: keyboard, unit price: ₹2500.00, stock: 8 units

Out: None

[Step 1: Duration 1.79 seconds| Input tokens: 2,220 | Output tokens: 123]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_quotation = {                                                                                              
      "Subtotal": "₹7500.00",                                                                                      
      "Discount": "5% (₹375.00)",                                                                                  
      "Total after discount": "₹7125.00",                                                                          
      "Shipping cost": "₹0.00",                                                                                    
      "Final amount": "₹7125.00"                                                                                   
  }                                                                                                                
  final_answer(final_quotation)                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'Subtotal': '₹7500.00', 'Discount': '5% (₹375.00)', 'Total after discount': '₹7125.00', 'Shipping 
cost': '₹0.00', 'Final amount': '₹7125.00'}

[Step 4: Duration 2.23 seconds| Input tokens: 10,135 | Output tokens: 427]


FINAL ANSWER
 {'Subtotal': '₹7500.00', 'Discount': '5% (₹375.00)', 'Total after discount': '₹7125.00', 'Shipping cost': '₹0.00', 'Final amount': '₹7125.00'}


### What happened internally?

The exact reasoning varies, but the observable loop should be similar to this:

1. Understand that product data is required.
2. Call `check_product`.
3. Observe the price and stock.
4. Call `calculate_order`.
5. Observe the discounted value.
6. Call `calculate_shipping`.
7. Combine verified observations into the final answer.

In [6]:
# Inspect the stored steps. The objects include model outputs, tool calls,
# observations, timing information, and errors when applicable.
steps = sales_agent.memory.steps
print(f"Number of stored memory steps: {len(steps)}")

for number, step in enumerate(steps, start=1):
    print(f"\n--- Memory step {number} ---")
    print(step)

Number of stored memory steps: 5

--- Memory step 1 ---
TaskStep(task='A customer in Kolkata wants 3 keyboards. Check stock, apply the correct discount, calculate shipping, and give a clear final quotation. Do not promise the order if stock is insufficient.', task_images=None)

--- Memory step 2 ---
ActionStep(step_number=1, timing=Timing(start_time=1787389229.8627625, end_time=1787389231.6572547, duration=1.794492244720459), model_input_messages=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, content=[{'type': 'text', 'text': 'You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.\nTo do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.\nTo solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.\n\nAt each step, in the \'Thought:\' sequence, you should first explain your reaso

## 6. CodeAgent vs ToolCallingAgent

| Feature | CodeAgent | ToolCallingAgent |
|---|---|---|
| Action format | Python code | Structured tool call |
| Best for | Calculations and multi-step transformations | Predictable, atomic tool dispatch |
| Flexibility | Higher | More controlled |
| Main concern | Generated code must be executed safely | Limited to predefined tool schemas |

The tools do not change; only the agent’s action mechanism changes.

In [7]:
dispatcher_agent = ToolCallingAgent(
    tools=[check_product, calculate_order, calculate_shipping],
    model=model,
    max_steps=8,
)

result = dispatcher_agent.run(
    "Check whether 2 webcams are available. If not, state that clearly."
)
print("\nFINAL ANSWER\n", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Check whether 2 webcams are available. If not, state that clearly.                                              │
│                                                                                                                 │
╰─ LiteLLMModel - openrouter/openai/gpt-4o-mini ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'check_product' with arguments: {'product_name': 'webcam'}                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Product: webcam, unit price: ₹3200.00, stock: 0 units

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'check_product' with arguments: {'product_name': 'webcam'}                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Both webcams are not available.'}                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Both webcams are not available.

Final answer: Both webcams are not available.

[Step 2: Duration 0.59 seconds| Input tokens: 2,741 | Output tokens: 67]


FINAL ANSWER
 Both webcams are not available.


## 7. Add conversational memory

By default, `run()` resets the agent for a new task. Pass `reset=False` when the next request should reuse the current conversation memory.

In [8]:
first_answer = dispatcher_agent.run(
    "What is the unit price and stock of a mouse?"
)
print(first_answer)

follow_up = dispatcher_agent.run(
    "Now calculate the order for 5 of them.",
    reset=False,
)
print(follow_up)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the unit price and stock of a mouse?                                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - openrouter/openai/gpt-4o-mini ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'check_product' with arguments: {'product_name': 'mouse'}                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Product: mouse, unit price: ₹1200.00, stock: 15 units

[Step 1: Duration 1.23 seconds| Input tokens: 1,291 | Output tokens: 15]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The total order value for 5 mice is ₹5400.00 with free │
│ shipping to Kolkata.'}                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The total order value for 5 mice is ₹5400.00 with free shipping to Kolkata.

Final answer: The total order value for 5 mice is ₹5400.00 with free shipping to Kolkata.

[Step 5: Duration 0.92 seconds| Input tokens: 7,393 | Output tokens: 123]

The total order value for 5 mice is ₹5400.00 with free shipping to Kolkata.


## 8. Build a multi-agent architecture

A multi-agent design is useful when responsibilities and tools can be separated cleanly. Here:

- the **inventory specialist** knows only how to check products,
- the **pricing specialist** knows only pricing and shipping, and
- the **manager agent** delegates work and combines the answers.

The specialists need clear `name` and `description` values because the manager uses them to decide whom to call.

In [15]:
inventory_specialist = ToolCallingAgent(
    tools=[check_product],
    model=model,
    name="inventory_specialist",
    description=(
        "Checks product availability and unit price. Give this agent the product "
        "name and required quantity."
    ),
    max_steps=4,
)

pricing_specialist = ToolCallingAgent(
    tools=[calculate_order, calculate_shipping],
    model=model,
    name="pricing_specialist",
    description=(
        "Calculates quantity discounts and delivery charges. Provide the unit "
        "price, quantity, and delivery city."
    ),
    max_steps=5,
)

manager_agent = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[inventory_specialist, pricing_specialist],
    max_steps=8,
)

multi_agent_answer = manager_agent.run(
    "Prepare a quotation for 4 monitors delivered to Mumbai. First confirm "
    "that enough stock exists. Show subtotal, discount, shipping, and final total."
)
print("\nMULTI-AGENT ANSWER\n", multi_agent_answer)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Prepare a quotation for 4 monitors delivered to Mumbai. First confirm that enough stock exists. Show subtotal,  │
│ discount, shipping, and final total.                                                                            │
│                                                                                                                 │
╰─ LiteLLMModel - openrouter/openai/gpt-4o-mini ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  product_name = "monitor"                                                                                         
  quantity = 4                                                                                                     
  stock_check = inventory_specialist(task="Check if there are enough monitors in stock for an order of 4 units.",  
  additional_args={"product_name": product_name, "quantity": quantity})                                            
  print(stock_check)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭──────────────────────────────────────── New run - inventory_specialist ─────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'inventory_specialist'.                                                            │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Check if there are enough monitors in stock for an order of 4 units.                                            │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│ You have been provided with these additional arguments, that you can access directly using the keys as          │
│ variables:                                                                                                      │
│ {'product_name': 'monitor', 'quantity': 4}.                                                                     │
│                                                                                                                 │
╰─ LiteLLMModel - openrouter/openai/gpt-4o-mini ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  quotation_details = {                                                                                            
      "Subtotal": 72000.00,                                                                                        
      "Discount": 7200.00,                                                                                         
      "Shipping": 500.00,                                                                                          
      "Final Total": 65300.00                                                                                      
  }                                                                                                                
                                                                                                                   
  final_answer(quotation_details)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'Subtotal': 72000.0, 'Discount': 7200.0, 'Shipping': 500.0, 'Final Total': 65300.0}

[Step 3: Duration 2.15 seconds| Input tokens: 8,129 | Output tokens: 538]


MULTI-AGENT ANSWER
 {'Subtotal': 72000.0, 'Discount': 7200.0, 'Shipping': 500.0, 'Final Total': 65300.0}


## 9. Single-agent or multi-agent?

| Choose a single agent when… | Choose multiple agents when… |
|---|---|
| The task and toolset are small | Specialists need different tools or instructions |
| One context is enough | Separate context reduces distraction |
| Low cost and latency matter most | Independent responsibilities are easy to define |
| Delegation adds no real value | A manager must coordinate different capabilities |

Do not create multiple agents only because it looks advanced. Each additional agent adds model calls, latency, cost, and another possible failure point.

## 10. Production checklist

Before moving an agent into a real application:

1. Give every tool one clear responsibility.
2. Validate tool inputs and handle failures explicitly.
3. Set `max_steps`, API timeouts, and spending limits.
4. Never put API keys directly in notebook source.
5. Log tool calls, observations, errors, token use, and latency.
6. Add human approval before payments, deletion, email, or other high-impact actions.
7. Treat external text as untrusted data, not as system instructions.
8. Use isolated execution for code agents and authorize only necessary imports.
9. Evaluate the agent with normal, ambiguous, adversarial, and failure cases.
10. Prefer a deterministic workflow when the sequence never needs model-based decisions.

## 11. Student exercises

### Exercise 1 — Add GST
Create a `calculate_gst(order_value, gst_rate)` tool and ask the agent for a tax-inclusive quotation.

### Exercise 2 — Add a product
Add a laptop to `PRODUCTS`, test the tool directly, and request a quotation.

### Exercise 3 — Test insufficient stock
Ask for 10 monitors. Check whether the agent refuses to promise an unavailable quantity.

### Exercise 4 — Compare architectures
Give the same quotation task to `CodeAgent` and `ToolCallingAgent`. Compare the number of steps, tool calls, clarity, and failure modes.

### Exercise 5 — Add a specialist
Create a customer-support specialist with a return-policy tool, then register it under the manager.

## 12. Key takeaways

- The **model** reasons; the **tools** perform actions; the **agent loop** coordinates them.
- `CodeAgent` is expressive because it composes actions with Python.
- `ToolCallingAgent` uses structured calls and is useful for controlled dispatch.
- `@tool` turns a well-described Python function into an agent capability.
- `reset=False` lets a later task reuse the agent’s existing memory.
- `managed_agents` enables hierarchical manager–specialist systems.
- The simplest architecture that reliably solves the task is usually the best starting point.

Official documentation: [Hugging Face smolagents](https://huggingface.co/docs/smolagents/index)